# 📡 Step 6 · Monitoring & Drift-Triggered Retraining

**Operations.** Watch the CLIP `embedding` feature for drift, then retrain the YOLOv8 detector automatically when production inputs shift.

`🗄️ image_embeddings → 📈 feature monitoring → 🚨 drift → 🔁 train_yolo`

This notebook adds two layers on top of the tutorial:

1. **Feature monitoring** on the `embedding` feature (centroid drift + norm-distribution PSI) and on a scalar feature (`num_bboxes`).
2. **Model monitoring** on the embeddings the deployed similarity service logs at inference time, with **drift-triggered retraining** of the detector via the existing `train_yolo` job.

> ℹ️ Run notebooks 1-5 and `predictor.ipynb` first, so `wider_face_files`, `image_embeddings`, the CLIP model, and the `similarimages` deployment all exist.

In [ ]:
import hopsworks

project = hopsworks.login()
fs = project.get_feature_store()
mr = project.get_model_registry()
job_api = project.get_jobs_api()
print(f"✅ Connected to project: {project.name}")

## 📈 Part A · Feature monitoring (drift visibility)

Feature-group monitoring resolves the detection and reference windows by commit-time travel, so no schema change is needed.

- **`embedding`** supports `centroid_distance` (L2 between window centroids) and distribution metrics over the per-row **norm** (PSI / KL / JS / Hellinger).
- **scalar features** use mean / PSI as usual.
- **distribution metrics over rolling windows** (`compare_on_distribution`) require `statistics_config.kll = True` on the feature group, so per-commit statistics runs persist the mergeable KLL sketch. Both groups enable it at creation (`create_fgs.py` for `wider_face_files`, notebook 4 for `image_embeddings`). For a group created before this change, retrofit once with `fg.statistics_config.kll = True; fg.update_statistics_config()`.

The feature to monitor is chosen on `.compare_on(...)` / `.compare_on_distribution(...)`, not on `create_feature_monitoring(...)`.

### 🧠 Embedding drift: centroid + norm PSI
Compare recent commits against a trailing 7-day baseline window.

In [ ]:
emb_fg = fs.get_feature_group("image_embeddings", version=1)

# Centroid drift on the embedding: recent commits vs a trailing baseline window
emb_fg.create_feature_monitoring(
    name="embedding_centroid_drift",
).with_detection_window(time_offset="1d") \
 .with_reference_window(time_offset="30d", window_length="7d") \
 .compare_on(feature_name="embedding", metric="centroid_distance", threshold=0.1).save()

# Norm-distribution drift (PSI over the per-row embedding L2 norm)
emb_fg.create_feature_monitoring(
    name="embedding_norm_psi",
).with_detection_window(time_offset="1d") \
 .with_reference_window(time_offset="30d", window_length="7d") \
 .compare_on_distribution(feature_name="embedding", metric="PSI", threshold=0.2).save()

print("✅ Embedding monitoring enabled: centroid_distance (>0.1) + norm PSI (>0.2)")

### 🔢 Scalar drift: faces per image
PSI on `num_bboxes` in the `wider_face_files` group.

In [ ]:
# Scalar example: drift in the number of faces per image
files_fg = fs.get_feature_group("wider_face_files", version=1)

files_fg.create_feature_monitoring(
    name="num_bboxes_drift",
).with_detection_window(time_offset="1d") \
 .with_reference_window(time_offset="30d", window_length="7d") \
 .compare_on_distribution(feature_name="num_bboxes", metric="PSI", threshold=0.2).save()

print("✅ Scalar monitoring enabled: num_bboxes PSI (>0.2)")

## 🤖 Part B · Model monitoring + drift-triggered retraining

`create_model_monitoring(...)` watches a deployed model's **inference logs**: it reads the feature view's logging feature group, filters by `model_name` / `model_version`, and needs the monitored model to carry a `training_dataset_version`.

`🖼️ query → 🧠 embedding → 📝 inference log → 📈 centroid drift → 🔁 train_yolo`

The served model (CLIP `similarimages`) and the retrained model (the YOLO detector) can differ, since `model_retraining_job` is any job. The wiring:

- 🗂️ a **logging-enabled feature view** over `embedding`, plus a baseline **training dataset** (the embedding population at indexing time) as the drift reference.
- 🏷️ a lightweight **monitor model** linked to that FV + TD, so monitoring has a `training_dataset_version`. The deployment tags each logged query embedding with it.
- 🔁 monitoring compares the logged embeddings' centroid against the baseline TD centroid. After 3 consecutive shifts it runs `train_yolo`, which re-fine-tunes and re-registers `facerecognition`.

### 🗂️ Feature view + baseline training dataset
A logging-enabled view over `embedding`, with the indexing-time population as the drift reference.

In [ ]:
# Logging-enabled feature view over the embedding feature
image_embeddings_fv = fs.get_or_create_feature_view(
    name="image_embeddings_fv",
    version=1,
    query=emb_fg.select(["embedding"]),
    logging_enabled=True,
)

# Baseline training dataset = the embedding population at indexing time (the drift reference)
td_version, _ = image_embeddings_fv.create_training_data(
    description="Baseline embedding population for drift monitoring",
)
print(f"✅ Feature view 'image_embeddings_fv' v{image_embeddings_fv.version} ready")
print(f"✅ Baseline training dataset version: {td_version}")

### 🏷️ Monitor model
A lightweight reference model linking the FV and baseline TD, so monitoring has a `training_dataset_version` and logged rows have a name to filter on.

In [ ]:
import os

# Lightweight reference ("monitor") model: links the embeddings feature view and the baseline
# training dataset so model monitoring has a recorded training_dataset_version. The deployed
# similarity service logs query embeddings under this model name.
monitor_dir = "/tmp/embeddings_monitor_model"
os.makedirs(monitor_dir, exist_ok=True)
with open(os.path.join(monitor_dir, "README.md"), "w") as f:
    f.write(
        "Reference model linking the image_embeddings feature view and the baseline training "
        "dataset, used as the monitored model for embedding drift detection."
    )

monitor_model = mr.python.create_model(
    name="image_embeddings_monitor",
    description="Reference model for embedding drift monitoring (links FV + baseline TD)",
    feature_view=image_embeddings_fv,
    training_dataset_version=td_version,
)
monitor_model.save(monitor_dir)
print(f"✅ Monitor model 'image_embeddings_monitor' v{monitor_model.version} registered")

### 📝 Logging predictor
Re-write the `similarimages` predictor so it logs each query embedding through the feature view, tagged with the monitor model. This is the only change needed to feed inference data to model monitoring.

In [ ]:
%%writefile predict_similar_images.py
import os
import io
import base64

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import hopsworks

# The query embeddings are logged under this reference ("monitor") model, which is linked
# to the embeddings feature view and the baseline training dataset (see notebook 6).
MONITOR_MODEL_NAME = "image_embeddings_monitor"


def get_image_embedding(image, processor, model, device):
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)
    # L2-normalize
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    return image_features.squeeze().cpu().tolist()


class Predict(object):
    def __init__(self, async_logger, model):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.project = hopsworks.login()
        self.fs = self.project.get_feature_store()

        # Resolve the latest monitor-model version so logged inference rows are tagged with
        # the same version the monitoring config filters on (get_model defaults to v1, so use
        # get_models and take the max version).
        mr = self.project.get_model_registry()
        self.monitor_model_version = max(
            m.version for m in mr.get_models(MONITOR_MODEL_NAME)
        )

        # Feature group (file_path + embedding) used for the similarity search
        self.fg = self.fs.get_feature_group("image_embeddings", version=1)

        # Logging-enabled feature view (embedding only) used for inference logging + monitoring
        self.fv = self.fs.get_feature_view("image_embeddings_fv", version=1)
        self.fv.init_serving(feature_logger=async_logger)

        model_path = os.environ["MODEL_FILES_PATH"]
        self.model = CLIPModel.from_pretrained(model_path).to(self.device).eval()
        self.processor = CLIPProcessor.from_pretrained(model_path)
        print("Initialization complete")

    def predict(self, inputs):
        try:
            b64_image = inputs[0][0]
            if not b64_image:
                return {"error": "Missing image (base64 string) in request"}

            image = Image.open(io.BytesIO(base64.b64decode(b64_image))).convert("RGB")
            embedding = get_image_embedding(image, self.processor, self.model, self.device)

            # Similarity search against the indexed embeddings
            results = self.fg.find_neighbors(embedding, k=3)
            returned_files, returned_images = [], []
            for result in results:
                path = result[1][0]
                returned_files.append(path)
                with open(path, "rb") as f:
                    returned_images.append(base64.b64encode(f.read()).decode("utf-8"))

            # Log the query embedding so model monitoring can detect drift in production inputs
            self.fv.log(
                untransformed_features=[[embedding]],
                model_name=MONITOR_MODEL_NAME,
                model_version=self.monitor_model_version,
            )

            return {"file_names": returned_files, "images": returned_images}
        except Exception as e:
            return {"error": str(e)}

### 🚀 Redeploy the similarity service
The served model stays the CLIP `openaiclip_vit_base_patch32`; only the predictor script changes (it now logs query embeddings).

In [ ]:
ms = project.get_model_serving()
model_mr = mr.get_model("openaiclip_vit_base_patch32", version=1)

dataset_api = project.get_dataset_api()
uploaded = dataset_api.upload(
    "predict_similar_images.py", model_mr.model_files_path, overwrite=True
)
script_path = os.path.join("/Projects", project.name, uploaded)

# Replace any existing deployment of this name with the logging-enabled one
existing = ms.get_deployment("similarimages")
if existing is not None:
    try:
        existing.stop(await_stopped=120)
    except Exception as e:
        print(f"(stop) {e}")
    existing.delete()

deployment = model_mr.deploy(name="similarimages", script_file=script_path)
deployment.start(await_running=300)
deployment.get_state().describe()

### 🔁 Retraining job
The retrain job is `train_yolo` (runs `train.py` in the `yolov8` GPU env). The cell reuses it if it exists, otherwise it registers it with the same configuration as `run-job.py`.

In [ ]:
import os

# The job's appPath must point at train.py in the project filesystem, and it must live in
# the tutorial folder itself (train.py reads sibling files: model.yaml, weights/, data/).
# With a mounted project filesystem the notebook's working directory ends with the folder's
# project-relative path, so probe cwd suffixes until one resolves.
dataset_api = project.get_dataset_api()

def _tutorial_dir_project_path():
    parts = os.getcwd().strip(os.sep).split(os.sep)
    for i in range(len(parts)):
        candidate = "/".join(parts[i:])
        if dataset_api.exists(f"{candidate}/train.py"):
            return candidate
    return None

# Retrain job: reuse the existing train_yolo, or register it (same config as run-job.py)
train_yolo_job = job_api.get_job("train_yolo")
if train_yolo_job is None:
    app_dir = _tutorial_dir_project_path()
    if app_dir is None:
        raise RuntimeError(
            "Could not locate this tutorial folder (train.py) in the project filesystem. "
            "Clone/upload the yolov8-face folder into the project (e.g. into the Jupyter "
            "dataset) and run this notebook from there."
        )
    print(f"Job 'train_yolo' not found, creating it (appPath: {app_dir}/train.py) ...")
    job_config = job_api.get_configuration("PYTHON")
    job_config["appPath"] = f"{app_dir}/train.py"
    job_config["environmentName"] = "yolov8"
    job_config["resourceConfig"]["cores"] = 1
    job_config["resourceConfig"]["memory"] = 10000
    job_config["resourceConfig"]["gpus"] = 1
    train_yolo_job = job_api.create_job("train_yolo", job_config)
train_yolo_job

### 🚨 Wire up drift-triggered retraining
Monitor the logged query embeddings for centroid drift; after 3 consecutive shifts, run `train_yolo`.

In [ ]:
# Monitor the logged query embeddings for centroid drift; after 3 consecutive shifts,
# run the train_yolo job (passes "retrain" as argv to train.py).
image_embeddings_fv.create_model_monitoring(
    name="detector_retrain_on_embedding_drift",
    model_name="image_embeddings_monitor",
    model_version=monitor_model.version,
    retrain_model_after_num_shifts=3,
    model_retraining_job=train_yolo_job,
    model_retraining_job_execution_args="retrain",
).with_detection_window(time_offset="1d") \
 .with_reference_training_dataset() \
 .compare_on(
     feature_name="embedding",
     metric="centroid_distance",
     threshold=0.1,
 ).save()

print("✅ Model monitoring enabled: retrain train_yolo after 3 consecutive centroid shifts (>0.1)")

## 🧪 Simulate embedding drift

To exercise the trigger without waiting for real drift, log a batch of "drifted" query embeddings (shifted away from the baseline) under the monitor model. The next monitoring runs see the detection-window centroid move past the threshold; after 3 consecutive shifts the `train_yolo` job runs.

In [ ]:
import numpy as np

# A real embedding to perturb away from
base = np.asarray(emb_fg.read().iloc[0]["embedding"], dtype=np.float64)
dim = base.shape[0]
rng = np.random.default_rng(42)

# Build drifted, L2-normalized query embeddings
drifted_rows = []
for _ in range(100):
    shifted = base + rng.normal(0.6, 0.2, size=dim)
    shifted = shifted / np.linalg.norm(shifted)
    drifted_rows.append([shifted.tolist()])

image_embeddings_fv.log(
    untransformed_features=drifted_rows,
    model_name="image_embeddings_monitor",
    model_version=monitor_model.version,
)
image_embeddings_fv.materialize_log(wait=True)
print("Logged drifted embeddings; monitoring runs will detect the shift.")

## 📝 Notes

- Thresholds (`centroid_distance` 0.1, PSI 0.2) and `retrain_model_after_num_shifts` (3) are starting points. CLIP embeddings are L2-normalized, so calibrate the centroid threshold against an observed no-drift baseline.
- This exercises the FSTORE-2048 embedding profiler + `centroid_distance` / norm-PSI path end to end.